# Model results: small CNN vs MobileNetV2

Trains both models from `model.py` on `data/kaggle/train`, checks them on `data/kaggle/validation`
after each epoch, and compares them on `data/kaggle/test`, on our own pictures in `data/test`, and live on the camera.

Every image is converted to grayscale, because the OV7675 camera only gives grayscale.
Trained weights go to `weights/` and training histories to `results/`. With `RETRAIN = False`,
models that were already trained are loaded instead of trained again.

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets

import model

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RETRAIN = False          # True: train again even if weights/ already has a model

EPOCHS_CNN = 15
EPOCHS_MOBILENET = 8
UNFREEZE_AT = 4          # epoch at which MobileNet's last blocks start training too

Path("weights").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
print("device:", DEVICE, "| torch", torch.__version__)
for split in ("train", "validation", "test"):
    ds = datasets.ImageFolder(model.KAGGLE / split)
    counts = np.bincount(ds.targets, minlength=len(ds.classes))
    print(f"{split:<10}", dict(zip(ds.classes, counts.tolist())))

device: cpu | torch 2.15.0+cpu
train      {'paper': 840, 'rock': 840, 'scissors': 840}
validation {'paper': 11, 'rock': 11, 'scissors': 11}
test       {'paper': 124, 'rock': 124, 'scissors': 124}


## What the models see
Training pictures after the random changes (top row) and evaluation pictures, which are only resized (bottom row).

In [2]:
def show_tensor(ax, x):
    x = x.clone()
    if x.shape[0] == 3:                      # undo the ImageNet normalization
        x = x * torch.tensor(model.IMAGENET_STD)[:, None, None] + torch.tensor(model.IMAGENET_MEAN)[:, None, None]
    else:
        x = x * 0.5 + 0.5
    ax.imshow(x[0].clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_axis_off()

train_ds = model.make_dataset("mobilenet", "train")
eval_ds = model.make_dataset("mobilenet", "test")
fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for i, ax in enumerate(axes[0]):
    x, y = train_ds[i * 400]
    show_tensor(ax, x); ax.set_title(model.CLASSES[y], fontsize=10)
for i, ax in enumerate(axes[1]):
    x, y = eval_ds[i * 60]
    show_tensor(ax, x); ax.set_title(model.CLASSES[y], fontsize=10)
plt.tight_layout(); plt.show()

: 

## Train
Each run keeps the weights from its best validation epoch. The Kaggle validation set has only 33 pictures,
so validation accuracy moves in steps of 3%; trust the test set more.

In [ ]:
def train_or_load(kind, build, epochs, lr, unfreeze_at=None):
    weights, hist_file = Path(f"weights/{kind}.pt"), Path(f"results/history_{kind}.json")
    net = build()
    if weights.exists() and hist_file.exists() and not RETRAIN:
        net.load_state_dict(torch.load(weights, map_location=DEVICE))
        print(f"{kind}: loaded {weights} (set RETRAIN = True to train again)")
        return net.to(DEVICE), json.loads(hist_file.read_text())
    loaders = model.make_loaders(kind)
    hist = model.train(net, loaders, epochs=epochs, lr=lr, device=DEVICE,
                       unfreeze_at=unfreeze_at, weights_path=weights)
    hist_file.write_text(json.dumps(hist))
    return net, hist

cnn, hist_cnn = train_or_load("cnn", model.smallCNN, EPOCHS_CNN, lr=1e-3)

In [ ]:
mobilenet, hist_mb = train_or_load("mobilenet", model.build_mobilenet, EPOCHS_MOBILENET, lr=3e-4,
                                   unfreeze_at=UNFREEZE_AT)

## Training curves

In [ ]:
runs = {"small CNN": hist_cnn, "MobileNetV2": hist_mb}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for (name, h), color in zip(runs.items(), ("tab:blue", "tab:orange")):
    ep = np.arange(1, len(h["train_acc"]) + 1)
    axes[0].plot(ep, h["train_acc"], "--", color=color, label=f"{name} train")
    axes[0].plot(ep, h["val_acc"], "-o", color=color, label=f"{name} validation")
    axes[1].plot(ep, h["train_loss"], "--", color=color, label=f"{name} train")
    axes[1].plot(ep, h["val_loss"], "-o", color=color, label=f"{name} validation")
axes[0].set(title="Accuracy", xlabel="epoch", ylim=(0, 1.02))
axes[1].set(title="Loss", xlabel="epoch")
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## Test set comparison

In [ ]:
models_ = {"small CNN": ("cnn", cnn), "MobileNetV2": ("mobilenet", mobilenet)}
results = {}
for name, (kind, net) in models_.items():
    loader = model.make_loaders(kind)["test"]
    t0 = time.perf_counter()
    y_true, y_pred, probs = model.predict_loader(net, loader, DEVICE)
    ms = 1000 * (time.perf_counter() - t0) / len(y_true)
    results[name] = dict(kind=kind, y_true=y_true, y_pred=y_pred, probs=probs, ms=ms)

def n_params(net, trainable=False):
    return sum(p.numel() for p in net.parameters() if p.requires_grad or not trainable)

hdr = f"{'':<13}{'test acc':>9}{'best val':>9}{'params':>11}{'s/epoch':>9}{'ms/image':>9}"
print(hdr); print("-" * len(hdr))
for name, r in results.items():
    h, net = runs[name], models_[name][1]
    acc = (r["y_true"] == r["y_pred"]).mean()
    print(f"{name:<13}{acc:>9.3f}{max(h['val_acc']):>9.3f}{n_params(net):>11,}"
          f"{np.mean(h['seconds']):>9.0f}{r['ms']:>9.1f}")
print("\nms/image includes loading the pictures from disk; a camera frame is faster.")

In [ ]:
present = sorted(set(np.concatenate([r["y_true"] for r in results.values()])))   # classes in the test set
fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 4.6))
for ax, (name, r) in zip(np.atleast_1d(axes), results.items()):
    cm = model.confusion(r["y_true"], r["y_pred"])[present]          # rows: true classes present
    ax.imshow(cm, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(len(model.CLASSES)), model.CLASSES)
    ax.set_yticks(range(len(present)), [model.CLASSES[c] for c in present])
    ax.set(xlabel="predicted", ylabel="true", title=f"{name}: test accuracy {(r['y_true'] == r['y_pred']).mean():.3f}")
plt.tight_layout(); plt.show()

print(f"{'':<13}{'class':<10}{'precision':>10}{'recall':>8}{'n':>6}")
for name, r in results.items():
    for c in present:
        tp = np.sum((r["y_pred"] == c) & (r["y_true"] == c))
        prec = tp / max(1, np.sum(r["y_pred"] == c))
        rec = tp / max(1, np.sum(r["y_true"] == c))
        print(f"{name:<13}{model.CLASSES[c]:<10}{prec:>10.3f}{rec:>8.3f}{np.sum(r['y_true'] == c):>6}")

## Mistakes
Up to 8 wrong test predictions per model, with the confidence of the wrong answer.

In [ ]:
for name, r in results.items():
    ds = model.make_dataset(r["kind"], "test")
    wrong = np.nonzero(r["y_true"] != r["y_pred"])[0]
    print(f"{name}: {len(wrong)} wrong out of {len(r['y_true'])}")
    if len(wrong) == 0:
        continue
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
    for ax in axes:
        ax.set_axis_off()
    for ax, i in zip(axes, wrong[:8]):
        show_tensor(ax, ds[i][0])
        p = r["y_pred"][i]
        ax.set_title(f"true {model.CLASSES[r['y_true'][i]]}\nsaid {model.CLASSES[p]} {r['probs'][i, p]:.0%}",
                     fontsize=9, color="tab:red")
    plt.show()

## Our own pictures (`data/test`)
The Kaggle pictures are computer-rendered hands on a white background. This is the real check: pictures from
our camera. It is skipped until `data/test` has pictures.

In [ ]:
own = Path("data/test")
n_own = sum(1 for p in own.rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg"))
if n_own == 0:
    print("data/test has no pictures yet: capture some into data/test/<class>/ and run this cell again.")
else:
    for name, (kind, net) in models_.items():
        ds = datasets.ImageFolder(own, transform=model.make_transform(kind, train=False), allow_empty=True)
        ds.target_transform = model._ToClasses(ds.classes)
        y_true, y_pred, _ = model.predict_loader(net, torch.utils.data.DataLoader(ds, batch_size=32), DEVICE)
        print(f"{name}: {(y_true == y_pred).mean():.3f} on {len(y_true)} of our pictures")
        print(model.confusion(y_true, y_pred))

## Live camera test
Selector S10 on **Camera**, jumpers J7 and J8 on the **Camera** end. Hold a gesture in the middle of the
picture (inside the yellow box). Interrupt the kernel to stop early.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
import ee105

def live_predict(seconds=30, width=160, height=120, scale=3):
    cam = ee105.connect_camera()
    font = ImageFont.load_default(size=15)
    handle = display(Image.new("L", (10, 10)), display_id=True)
    times = []
    try:
        t_end = time.perf_counter() + seconds
        while time.perf_counter() < t_end:
            img = cam.snap(width, height)
            times.append(time.perf_counter())
            lines = []
            for name, (kind, net) in models_.items():
                label, p = model.predict_frame(net, kind, img, device=DEVICE)
                lines.append(f"{name}: {label} {p.max():.0%}")
            frame = Image.fromarray(img).resize((width * scale, height * scale), Image.NEAREST).convert("RGB")
            d = ImageDraw.Draw(frame)
            s = int(round(min(width, height) * 0.8)) * scale
            x0, y0 = (frame.width - s) // 2, (frame.height - s) // 2
            d.rectangle([x0, y0, x0 + s, y0 + s], outline=(255, 200, 0), width=2)
            fps = (len(times[-10:]) - 1) / (times[-1] - times[-10:][0]) if len(times) > 1 else 0
            for k, text in enumerate(lines + [f"{fps:.1f} fps"]):
                d.text((6, 4 + 18 * k), text, fill=(255, 255, 0), font=font, stroke_width=2, stroke_fill=(0, 0, 0))
            handle.update(frame)
    except KeyboardInterrupt:
        pass
    finally:
        cam.close()

live_predict(seconds=30)